# DRAGNET — deployment-scale probe: the structure at k = 30 (Kaggle)

The grid enumerates sufficient-set families at k = 6; the sharpest open question is whether
the plural / disjoint / non-monotone structure survives at deployment depth, where
enumeration is impossible. This builds a natural cell at **k = 20** (3.3x the grid; the most
a dual-T4 fits — a 30-passage prefill OOMs even sharded) whose distractors are a live BM25
retriever's top hits over the corpus, then certifies the three properties per wrong case with
two grow-prune passes and a few sampled supersets — O(k) queries per case, lower bounds by
construction, never the 2^k lattice.

**Before running:** Settings → Accelerator **GPU T4** (one card is enough — 4-bit fits a 16 GB
card; P100 also works) · Internet **on**.

Rough budget: build ~10–20 min (it stops once it has enough wrong cases, not all 400), probe
~2–4 h at 60 cases. 4-bit on one card is far faster than an fp16 model sharded across two (which
pays a cross-device hop per layer). An existing cell is reused, so an interrupted session
resumes at the probe. Everything lands in `runs/` and zips at the end.

In [ ]:
import os, shutil, subprocess, sys
from pathlib import Path

WORK = Path('/kaggle/working')
os.chdir(WORK)

# Long prompts fragment CUDA memory (reserved-but-unallocated blocks the allocator won't reuse);
# let it reclaim them. Set in the parent env so the probe subprocess inherits it.
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'

def fetch(name, url):
    if (WORK / name).exists():
        subprocess.run(['git', '-C', name, 'pull', '--ff-only'], check=True)
    else:
        subprocess.run(['git', 'clone', '--depth', '1', url, name], check=True)

fetch('lineup', 'https://github.com/santoshcheethiralame-dot/LINEUP')

# an attached repo dataset (if any) takes precedence over the clone, for offline runs.
dragnet_src = next((p.parent for p in Path('/kaggle/input').glob('*/**/src/dragnet/__init__.py')), None)
if dragnet_src is not None:
    shutil.rmtree(WORK / 'dragnet', ignore_errors=True)
    shutil.copytree(dragnet_src.parent.parent, WORK / 'dragnet')
else:
    fetch('dragnet', 'https://github.com/santoshcheethiralame-dot/DRAGNET')

subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', './lineup[gpu]'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--no-deps', '-e', './dragnet'], check=True)
# The container's transformers refuses 4-bit below this floor; pin it regardless of what resolved.
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-U', 'bitsandbytes>=0.46.1'], check=True)

# In the notebook kernel a bare `import lineup` binds to the cloned folder (a namespace package
# with no submodules) and shadows the pip install -> `lineup.data` fails. Real package dirs first.
for pkg in ('lineup', 'dragnet'):
    src = str(WORK / pkg / 'src')
    if src not in sys.path:
        sys.path.insert(0, src)

import bitsandbytes
print('[ok] lineup + dragnet installed; bitsandbytes', bitsandbytes.__version__)

In [ ]:
MODEL = 'Qwen/Qwen2.5-7B-Instruct'
TAG = 'qwen'
DATASET = 'hotpotqa'          # hotpotqa | 2wiki | musique
K = 20                        # retrieval depth. 20 is the largest a dual-T4 fits: a 30-passage
                              # prefill OOMs even sharded fp16 (~7GB activation/card). Still 3.3x
                              # the main grid's k=6 and the low end of the reviewer's 20-50 band
LIMIT = 400                   # source questions; the organic error rate at this depth sets n
LIMIT_CASES = 60              # wrong cases to probe (0 = all); the build stops once it has this
                              # many plus a small surplus, so it never generates on all 400
NONMONO_SAMPLES = 4           # sampled supersets per case for the non-monotonicity check
SEED = 0

cell = f'runs/{DATASET}/largek{K}/{TAG}'
print(cell)

In [ ]:
# Build the k=30 BM25-natural cell (reused if present), then run the bounded probe.
# Progress streams live: build every 25 questions, probe every 10 cases with running rates.
import subprocess, sys

subprocess.run(
    [sys.executable, 'dragnet/scripts/run_largek_probe.py', '--cell', cell,
     '--dataset', DATASET, '--k', str(K), '--limit', str(LIMIT), '--seed', str(SEED),
     '--model', MODEL, '--load-in-4bit',
     '--limit-cases', str(LIMIT_CASES), '--nonmono-samples', str(NONMONO_SAMPLES)],
    check=True,
)

In [ ]:
# Quick read of the probe artifact: rates with Wilson intervals beside the k=6 grid values.
# Guarded so a hiccup here can never stop the zip — the artifact is already on disk.
try:
    import json
    from collections import Counter
    from math import sqrt

    rows = [json.loads(l) for l in open(f'{cell}/largek_probe.jsonl', encoding='utf-8') if l.strip()]
    ok = [r for r in rows if r['status'] == 'ok']

    def wilson(hits, n, z=1.96):
        p, d = hits / n, 1 + z * z / n
        centre, margin = p + z * z / (2 * n), z * sqrt(p * (1 - p) / n + z * z / (4 * n * n))
        return (centre - margin) / d, (centre + margin) / d

    n = len(ok)
    print(f"k={K}  wrong probed {len(rows)}  evaluable {n}  "
          f"parametric {sum(r['status'] == 'parametric' for r in rows)}  "
          f"budget-exhausted {sum(r['status'] == 'budget' for r in rows)}")
    for key, ref in (('plural', '0.45-0.63'), ('disjoint', '0.28'), ('nonmono', '0.67-0.87')):
        hits = sum(r[key] for r in ok)
        lo, hi = wilson(hits, n)
        print(f"  {key:8s} >= {hits / n:.2f}  [{lo:.2f}, {hi:.2f}]   (k=6 grid: {ref})")
    print('  certified set sizes:', dict(sorted(Counter(len(r['mss1']) for r in ok).items())))
except Exception as exc:
    print('readback skipped:', repr(exc))

In [ ]:
# Zip everything for download, self-labelled.
import shutil
stem = f'dragnet_{DATASET}_{TAG}_largek{K}_seed{SEED}'
shutil.make_archive(stem, 'zip', 'runs')
print('download:', f'/kaggle/working/{stem}.zip')